# Experiment 2.1 (staged) — Does misconception-matched context help generation?

**Why this, and not G2 vs G3 yet.** Measured on the frozen retrievers, MNRL and pretrained MiniLM return **64% identical** top-5 exemplars and differ in matched-exemplar status for only **10/187 questions**. The expected G3−G2 generation gap is ≈0.7pp against a ≈5.3pp detection threshold at n=187 — roughly **7× underpowered**. Running it would yield an uninformative null.

This experiment instead maximises the contrast by construction, testing the **precondition** for retrieval quality to matter at all.

| Arm | Exemplars | Matched |
|---|---|---|
| **G1** Random | 5 random corpus QDPs | ~0.01/5 |
| **G4** Oracle | 5 sharing a gold misconception | **5.00/5** |
| **G5** Anti-oracle | 5 same-subject, none matching | **0.00/5** |

G4/G5 consult gold labels — they are **diagnostics, not deployable systems**. G5 is drawn from the same subject as the target, so the G4/G5 contrast isolates *misconception match* rather than topical similarity. G1 is the unconstrained reference.

**No retriever is needed** — all three arms select exemplars from metadata, so there is no 15-minute retrain.

**Primary comparison:** G4 vs G5 on exact-match, pre-specified. All others are secondary and Holm-corrected.

Set `REPO_URL`, Runtime → GPU, Run all. ~1 GPU-hour.

In [ ]:
# --- The ONLY cell you edit ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- set this
SEED = 42

In [ ]:
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
import os
if not os.path.exists("distractor"):
    !git clone {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt
if not os.path.exists("outputs/results/train_qdp.csv"):
    !python scripts/01_prepare_dataset.py
print("Ready.")

## Step 1 — Manipulation check + prompt inspection (no GPU cost)

Confirms the arms differ exactly as intended before any generation. Expect G4 ≈ 5.00 matched, G5 = 0.00, G1 ≈ 0.01.

In [ ]:
!python scripts/09_run_mechanism.py --dry-run

## Step 2 — Smoke test (5 questions per arm)

First load of Qwen2.5-7B in 4-bit. If it OOMs on a T4, add `--llm Qwen/Qwen2.5-3B-Instruct`.

In [ ]:
!python scripts/09_run_mechanism.py --limit 5 --gen-seed {SEED}

## Step 3 — Full run (~1 GPU-hour)

Writes one JSONL per arm, appending after every question. **If Colab disconnects, just re-run this cell** — completed questions are skipped.

In [ ]:
!python scripts/09_run_mechanism.py --gen-seed {SEED}

## Step 4 — Evaluate

In [ ]:
!python scripts/10_evaluate_mechanism.py

In [ ]:
print(open("outputs/results/exp21/exp21_report.md").read())

In [ ]:
from IPython.display import Image, display
display(Image("outputs/figures/exp21/exp21_arms.png"))

## How to read the result

**G4 significantly > G5** → misconception-matched context causally improves generation. Retrieval quality has real headroom, and the G4−G5 delta approximates the *maximum* any retriever could deliver at k=5. Proceed to the full G0/G1/G2/G3 comparison, but size it against that ceiling — if the ceiling is small, even a perfect retriever will not produce a detectable G2/G3 difference.

**G4 ≈ G5 (null)** → even perfect retrieval does not measurably improve generation under this prompt. This is a strong, publishable negative finding: it means Stage 1's retrieval gains do **not** transfer to Stage 2, and the G2/G3 comparison is not worth running. Effort should redirect to the prompt design or the generator.

**G5 ≈ G1** → topical relevance alone contributes nothing; only misconception match matters.

**Caveats to carry into the write-up.** The primary contrast runs on the ~110-question subset where both arms are constructible at k=5, which over-represents commoner misconceptions and under-represents the long tail. `misconception_similarity` reflects the model's *self-reported* intent, not verified behaviour. Exact-match is binary and low-powered; when it disagrees with the continuous metrics, weight the continuous ones.

In [ ]:
!zip -qr exp21_results.zip outputs/generation/exp21 outputs/results/exp21 outputs/figures/exp21
from google.colab import files
files.download("exp21_results.zip")